# Démo - Ollama

Ollama est un outil qui permet d’exécuter des modèles de langage directement sur une machine locale. Contrairement à plusieurs solutions en ligne, il ne nécessite pas de connexion à un serveur externe une fois les modèles téléchargés. Cela permet de travailler hors ligne, de mieux contrôler les données utilisées et de réduire les délais liés aux appels réseau.

Ollama fournit une interface simple en Python avec la fonction chat(), qui permet d’interagir avec un modèle sous forme de conversation. Le modèle reçoit une liste de messages (rôles user, assistant, tool) et génère une réponse structurée. Il peut également proposer l’utilisation d’outils externes (fonctions Python), ce qui permet de combiner raisonnement et exécution de code.

Lien pour télécharger Ollama : 

- https://ollama.com/download/

Lien pour installer et utiliser Ollama : 

- https://docs.ollama.com/linux
- https://docs.ollama.com/windows


Liens pour télécharger et utiliser Gemma 4 sur Ollama:

- https://ollama.com/library/gemma4

Les modèles de Gemma 4 qui peuvent être utilisés (par exemple):

- `model="gemma4:e2b"` (2 milliards de paramètres)
- `model="gemma4:e4b"` (4 milliards de paramètres)
- `model="MODEL_COURANT"` (26 milliards de paramètres)



**Ollama vs HuggingFace**

Ollama et Hugging Face permettent tous deux d’utiliser des modèles de langage, mais leur approche diffère.

Ollama se distingue par sa **simplicité d’utilisation** et sa **rapidité**. Les modèles sont souvent quantifiés, ce qui réduit leur taille et accélère leur exécution, surtout sur CPU. Il offre aussi un **support intégré pour les Outils**, permettant au modèle de proposer directement l’utilisation de fonctions externes.

Hugging Face, de son côté, est plus flexible et complet, mais demande davantage de configuration. 

##  Importation des modules

Modules qui doivent potentiellement être installé:

- `librosa`
- `ollama`
- `accelerate`
- `sacremoses`

In [2]:
from ollama import chat
from pprint import pprint
import time
from datetime import date
import requests



## Exercice 1 - Analyse d'image

On analyse une image de chats (`chat.png`). Gemma répond en français, même si la question est en anglais, puisque c'est spécifié avec le rôle `system`.

Notez que la première exécution peut être plus lente, puisque le modèle doit être mis en mémoire la première fois.

- Le modèle est gardé en mémoire environ 5 minutes (valeur par défaut)
- Il est possible de changer cette valeur avec le paramètre `keep_alive=300` qui donne le temps en secondes (`-1` pour infinie)

In [ ]:
MODEL_COURANT = "gemma4:31b-cloud"

debut = time.perf_counter()

response = chat(
    model=MODEL_COURANT,
    messages=[
                {
            "role": "system",
            "content": "Répond en français",
        },
        {
            "role": "user",
            "content": "Write single detailed caption for this image.",
            "images": ["cat.jpg"]
        }
    ],
    think=False,
    keep_alive=3000
)

fin = time.perf_counter()

print(f"\n\n{"="*50}")
print(f"Temps   : {(fin-debut):.2f} s")
pprint(response["message"]["content"])



ResponseError: model 'MODEL_COURANT' not found (status code: 404)

## Exercice 2 - Le thinking

Génération de texte SANS thinking.

Ici, toute la réponse de Gemma est affichée. Observez les différentes valeurs qui sont fournies.

Notamment ce qui est dans le `message`:

- `content`: Le message lui-même
- `role`: De qui vient le message (ici, c'est de l'`assistant`, soit Gemma)
- `thinking`: Le "raisonnement" de Gemma pour obtenir la réponse
- `tool_calls`: les outils que Gemma veut appeler (on voit ça plus bas)

Les autres informations indiquent, entre-autre, le temps d'exécution et le nombre de **tokens** utilisés (`eval_count` et `prompt_eval_count`)

In [ ]:


debut = time.perf_counter()

response = chat(
    model="MODEL_COURANT",
    messages=[
        {
            "role": "user",
            "content": "Explique ce qu'est un transformer en 2 phrases"
        }
    ], 
    think=False, 
    keep_alive=3000
)

fin = time.perf_counter()

print(f"\n\n{"="*50}")
print(f"Temps   : {(fin-debut):.2f} s")
pprint(response.model_dump())

### Exercice 2.1 

Génération de texte AVEC thinking.

Notez la différence de temps d'exécution et la valeur du `thinking` de la réponse.

In [ ]:

debut = time.perf_counter()

response = chat(
    model="MODEL_COURANT",
    messages=[
        {
            "role": "user",
            "content": "Explique ce qu'est un transformer en 2 phrases"
        }
    ], 
    think=True
)

fin = time.perf_counter()

print(f"\n\n{"="*50}")
print(f"Temps   : {(fin-debut):.2f} s")
pprint(response.model_dump())

## Exercice 3 - La mémoire de conversation

Les modèles n'ont pas de mémoire. Si on veut avoir une conversation, il faut garder l'historique de la conversation et envoyer le tout à chaque prompt.

Ici, on garde cette historique. 

Si on utilise le `thinking`, il est important de ne pas le mettre dans l'historique.

In [ ]:

messages = [
    {"role": "system", 
    "content":"Réponds brièvement et clairement en français. \
                    Si on te demande de l'aide, n'hésite pas à demander des informations précise pour donner la réponse\
                    Si tu as une réponse précise, donne-la dès le début, avant de donner les détails de la démarche"},
]


while True:
    entree = input("Entrez le texte")
    if len(entree) == 0:
        break

    messages.append({
        "role": "user", "content": entree
        }
    )

    debut = time.perf_counter()
    resultat = chat("MODEL_COURANT", messages=messages, think=False, keep_alive=3000)
    texte_reponse = resultat["message"]["content"]
    fin = time.perf_counter()

    print(f"\n\n{"="*50}")
    print(f"Temps   : {(fin-debut):.2f} s")
    print(f">>>>>>> : {entree}")
    print(f"Réponse : {texte_reponse}")


    messages.append(
        {
            "role": "assistant",
            "content":  texte_reponse
        }
    )



## Exercice 4 - Traduction de texte

Pour traduire un texte, il suffit simplement de le demander dans le prompt.

Notez le temps d'exécution par rapport aux modèles spécialisés.

In [ ]:

texte_anglais = "Steven Spielberg, a fan of the comics and toys,[5] signed on as executive producer in 2004. John Rogers wrote the first draft, which pitted four Autobots against four Decepticons,[9] and featured the Ark spaceship.[10] Roberto Orci and Alex Kurtzman, fans of the cartoon,[11] were hired to rewrite the script in February 2005.[12] Spielberg suggested that a boy and his car should be the focus.[13] This appealed to Orci and Kurtzman because it conveyed themes of adulthood and responsibility, 'the things that a car represents in the United States'.[14] The characters of Sam and Mikaela were the sole point of view given in Orci and Kurtzman's first draft.[15] The Transformers had no dialogue, as the producers feared talking robots would look ridiculous. The writers felt that even if it would look silly, not having the robots speak would betray the fanbase.[11] The first draft also had a battle scene in the Grand Canyon.[16] Spielberg read each of Orci and Kurtzman's drafts and gave notes for improvement.[13] The writers remained involved throughout production, adding additional dialogue for the robots during the sound mixing (although none of this was kept in the final film, which ran fifteen minutes shorter than the initial edit).[17] Furman's The Ultimate Guide, published by Dorling Kindersley, remained as a resource to the writers throughout production.[17] Prime Directive was used as a fake working title. This was also the name of Dreamwave Productions' first Transformers comic book.[18]"

debut = time.perf_counter()

response = chat(
    model="MODEL_COURANT",
    messages=[
        {
            "role": "user",
            "content": f"Traduit moi ce texte : {texte_anglais}"
        }
    ], 
    think=False, 
    keep_alive=3000
)

fin = time.perf_counter()

print(f"\n\n{"="*50}")
print(f"Temps   : {(fin-debut):.2f} s")
pprint(response["message"]["content"])


## Exercice 5 - Compléter une phrase

Comme pour la traduction, il suffit de demander dans le prompt ce qu'on veut.

In [ ]:
# Compléter une phrase

texte_anglais = "The capital of Canada is"

debut = time.perf_counter()


response = chat(
    model="MODEL_COURANT",
    messages=[
        {
            "role": "user",
            "content": f"Complete la phrase suivante : {texte_anglais}"
        }
    ], 
    think=False, 
    keep_alive=3000
)

fin = time.perf_counter()

print(f"\n\n{"="*50}")
print(f"Temps   : {(fin-debut):.2f} s")
pprint(response["message"]["content"])

## Exercice 6 - Mise en contexte

On utilise le role `system` pour mettre en contexte.

Est-ce que le temps de génération de texte vous semble adéquat ?



In [ ]:
# Mise en contexte avec le role system

debut = time.perf_counter()

response = chat(
    model="MODEL_COURANT",
    messages=[
        {
            "role": "system",
            "content": "Répond en français. En 200 mots"
        },
        {
            "role": "user",
            "content": "Combien y a t'il de neurones dans le cerveau humain ?"
        }
    ], 
    think=False, 
    keep_alive=3000
)

fin = time.perf_counter()

print(f"\n\n{"="*50}")
print(f"Temps    : {(fin-debut):.2f} s")
print(f"Longueur : {len(response["message"]["content"].split(" "))}")
print(response["message"]["content"])

## Exercice 7 - Transcription de texte

On peut extraire du texte à partir d'une image

In [ ]:
# Transcription du texte sur image

messages = [
    {
        "role": "user",
        "images": ["texte.png"],
        "content": "Peux-tu me transcrire le texte qui se trouve sur l'image"
    },
]

debut = time.perf_counter()
response = chat(model="MODEL_COURANT", messages=messages, think=False, keep_alive=3000)
texte_reponse = response["message"]["content"]
fin = time.perf_counter()

print(f"\n\n{"="*50}")
print(f"Temps    : {(fin-debut):.2f} s")
print(f"Longueur : {len(texte_reponse.split(" "))}")
print(f"Réponse  : {texte_reponse}")



## Exercice 8 - Utilisation des outils

### Exercice 8.1 - Équation mathématique

#### Exercice 8.1.1 - SANS outil

On demande à Gemma de résoudre une équation relativement simple...

Est-ce que la réponse est juste ?

In [ ]:

equation = "3625965412 * 63256896 + 632512"
# equation = "10 * 36 + 10 + 36 * 31"

messages = [
    {
        "role": "user",
        "content": f"Donne moi la réponse (seulement la réponse) de l'équation suivante : {equation}"
    },
]


debut = time.perf_counter()
response = chat(model="MODEL_COURANT", messages=messages, think=False, keep_alive=3000)
texte_reponse = response["message"]["content"]
fin = time.perf_counter()

print(f"\n\n{"="*50}")
print(f"Temps   : {(fin-debut):.2f} s")
print(f"Réponse : {texte_reponse}")


print(f"Réponse : {eval(equation)} (bonne réponse)")

#### Exercice 8.1.2 - Utilisation d'outil avec le prompt seulement 

On demande à Gemma de nous indiquer s'il faut résourdre une équation. Si c'est le cas, il nous réponds avec les balises <equation>.

Si c'est le cas, nous parsons la réponse et trouvons la réponse avec une méthode plus efficace (`eval()` en python).

Notez que Ollama permet d'utliser une méthode plus efficace (prochain exercice). Mais avec HuggingFace, il faudrait utiliser une méthode comme celle-ci.

In [ ]:

equation = "3625965412 * 63256896 + 632512"
# equation = "10 * 36 + 10 + 36 * 31"

messages = [
    {
        "role": "system",
        "content": "Si on te donne une équation à résoudre, retourne plutot seulement l'équation \
                qui pourra être résolue avec eval() dans python pour trouver la réponse.\
                    Met la balise <equation> et </equation> à ta réponse dans ce cas. " 
    },
    {
        "role": "user",
        "content": f"Donne moi la réponse (seulement la réponse) de l'équation suivante : {equation}"
    },
]


debut = time.perf_counter()
response = chat(model="MODEL_COURANT", messages=messages, think=False, keep_alive=3000)
texte_reponse = response["message"]["content"]


if texte_reponse.startswith("<equation>"):
    texte_reponse = texte_reponse.replace('<equation>', "")
    texte_reponse = texte_reponse.replace('</equation>', "")
    print(f"EQUATION.... : {texte_reponse}")
    texte_reponse = eval(texte_reponse)
    
fin = time.perf_counter()

print(f"\n\n{"="*50}")
print(f"Temps   : {(fin-debut):.2f} s")
print(f"Réponse : {texte_reponse}")


print(f"Réponse : {eval(equation)} (bonne réponse)")

#### Exercice 8.1.3 - Avec `tools`

Maintenant, nous utiliseront une méthode fournie par Ollama.

Dans un premier temps, nous définissons une fonction `calculatrice()`, qui permet de résourdre une équation.

Ensuite, nous créons la liste `outils` qui contient les outils que Gemma pourra utilisés s'il pense qu'il en a besoin. Pour le moment, on a seulement l'outil de calculatrice.

- La description permet à Gemma de comprendre ce que l'outil peut faire
- Les paramètres sont ce que Gemma retournera pour utiliser l'outil
  - Dans notre exemple, il retournera l'équation qui devra être passée à `calculatrice()`

Finalement, on utilise `chat()`, mais en ajoutant le paramètre `tools`.


Observez la réponse!

Notamment, le `content` et `tool_calls` du `message`.

In [ ]:

def calculatrice(equation):
    return eval(equation)


outils = [
    {
        "type": "function",
        "function": {
            "name": "calculatrice",
            "description": "Donne la réponse d'une équation mathématique",
            "parameters": {
                "type": "object",
                "properties": {
                    "equation": {"type": "string"}
                },
                "required": ["equation"]
            }
        }
    }
]


equation = "3625965412 * 63256896 + 632512"
# equation = "10 * 36 + 10 + 36 * 31"

messages = [
        {"role": "user", "content": f"Quelle est la réponse de l'équation suivante : {equation}"}
    ]

response = chat(model="MODEL_COURANT", messages=messages, tools=outils, think=False, keep_alive=3000)

pprint(response.model_dump())

##### Exercice 8.1.3.1

La réponse n'a pas de contenu. Mais elle demande d'utiliser l'outil `calculatrice`.

Donc, suite à ce type de réponse, on appelle la fonction appropriée (`calculatrice()` dans notre cas) avec les bons arguements (fournis par Gemma).

La réponse de l'outil est ensuite rajoutée à la suite des messages avec le rôle `tool`. Comme nous faisions dans l'exercice avec la mémoire.

Les messages sont repassés à Gemma, qui nous donnera la réponse finale.

In [ ]:
resultat_outil = ""

if response.message.tool_calls:
    for tool in response.message.tool_calls:
        if tool.function.name == "calculatrice":
            resultat_outil = calculatrice(tool.function.arguments["equation"])

print("Outil : ", resultat_outil)

if resultat_outil != "":
    messages = [
        {"role": "user", "content": f"Quelle est la réponse de l'équation suivante : {equation}"},
        response.message,
        {"role": "tool", "content": f"{resultat_outil}"}
    ]

    response2 = chat(model="MODEL_COURANT", messages=messages, tools=outils, think=False, keep_alive=3000)

    pprint(response2.model_dump())
    

print(f"Bonne Réponse : {eval(equation)}")

### Exercice 8.2 - La date d'aujourd'hui

Les modèles n'ont pas la capacité de savoir la date actuelle.

Nous pouvons donc utiliser un outil pour l'aider.

Voici un exemple sans outil :

In [ ]:

messages = [
        {"role": "user", "content": f"Quel jour sommes-nous ?"}
    ]

response = chat(model="MODEL_COURANT", messages=messages, tools=outils, think=False, keep_alive=3000)

pprint(response.model_dump())

#### Exercice 8.2.1 - Ajout de l'outil `date_actuelle`

Notez que cet outil n'a pas besoin de paramètre.

Ici, on définie la fonction `utiliser_outils()` pour simplier l'utilisation des outils (nous en auront plusieurs). Les affichages dans cette fonction permettent de voir quels outils est utilsé, mais ne sont pas nécessaires lors de l'utilisation finale.

In [ ]:
# on ajoute des outils


def date_actuelle():
    return date.today()


outils = [
    {
        "type": "function",
        "function": {
            "name": "calculatrice",
            "description": "Donne la réponse d'une équation mathématique",
            "parameters": {
                "type": "object",
                "properties": {
                    "equation": {"type": "string"}
                },
                "required": ["equation"]
            }
        }
    },
    {
            "type": "function",
            "function": {
                "name": "date_actuelle",
                "description": "Donne la date de la journée actuelle",
            }
        }
]



def utiliser_outils(response, messages):
    print("OUTIL : ")
    resultat_outil = ""
    for tool in response.message.tool_calls:
        if tool.function.name == "calculatrice":
            print(">> équation")
            resultat_outil += str(calculatrice(tool.function.arguments["equation"]))
        if tool.function.name == "date_actuelle":
            print(">> date")
            resultat_outil += str(date_actuelle())
        resultat_outil += "\n"

    print("\n", "*"*20)
    print(resultat_outil)
    print("\n", "*"*20)

    messages.append(response.message)
    messages.append(
        {"role": "tool", "content": f"{resultat_outil}"}
    )

    return messages
    




#### Exercice 8.2.2 

Ici, si la réponse du modèle contient des demande d'appels d'outils (`tool_calls`), on appelle la fonction `utiliser_outils()`.

In [ ]:




messages = [
        {"role": "user", "content": f"Quel jour sommes-nous ?"}
    ]

response = chat(model="MODEL_COURANT", messages=messages, tools=outils, think=False, keep_alive=3000)


if response.message.tool_calls:
    messages = utiliser_outils(response, messages)
    response = chat(model="MODEL_COURANT", messages=messages, tools=outils, think=False, keep_alive=3000)


pprint(response.model_dump())

### Exercice 8.3 - La Météo

La fonction `meteo_demain()` fournie des informations concernant la météo prévue selon les coordonnées géographiques.  
La fonction utilise un API.


In [ ]:

def meteo_demain(latitude, longitude):
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum,precipitation_probability_max",
        "timezone": "auto"
    }
    data = requests.get(url, params=params).json()
    pluie = data["daily"]["precipitation_sum"][1]
    prob = data["daily"]["precipitation_probability_max"][1]
    maxi = data["daily"]["temperature_2m_max"][1]
    mini = data["daily"]["temperature_2m_min"][1]
    
    reponse = f"température max : {maxi},  température min : {mini},  pluie : {pluie} mm,  probabilité de pluie : {prob} %"
    print(f"latitude : {latitude},   longitude: {longitude}")
    return reponse

meteo_demain(45.5, -73.6)

#### Exercice 8.3.1 - Création de l'outil Météo

Donc, on rajoute l'outil de météo à la liste des outils disponibles.

Il faut aussi ajuster la fonction `utiliser_outils()` qui soit savoir quoi faire avec le nouvel outil.

In [ ]:

outils = [
    {
        "type": "function",
        "function": {
            "name": "calculatrice",
            "description": "Donne la réponse d'une équation mathématique",
            "parameters": {
                "type": "object",
                "properties": {
                    "equation": {"type": "string"}
                },
                "required": ["equation"]
            }
        }
    },
    {
            "type": "function",
            "function": {
                "name": "date_actuelle",
                "description": "Donne la date de la journée actuelle",
            }
    },
    {
        "type": "function",
        "function": {
            "name": "meteo_demain",
            "description": "Donne la météo à l'endroit indiqué par la latitude et longitude",
            "parameters": {
                "type": "object",
                "properties": {
                    "latitude": {"type": "string"},
                    "longitude": {"type": "string"},
                },
                "required": ["latitude", "longitude"]
            }
        }
    }
]


def utiliser_outils(response, messages):
    print("OUTIL : ")
    resultat_outil = ""
    for tool in response.message.tool_calls:
        if tool.function.name == "calculatrice":
            print(">> équation")
            resultat_outil += str(calculatrice(tool.function.arguments["equation"]))
        if tool.function.name == "date_actuelle":
            print(">> date")
            resultat_outil += str(date_actuelle())
        if tool.function.name == "meteo_demain":
            print(">> meteo")
            resultat_outil += meteo_demain(tool.function.arguments["latitude"], tool.function.arguments["longitude"])
        resultat_outil += "\n"

    print("\n", "*"*20)
    print(resultat_outil)
    print("\n", "*"*20)

    messages.append(response.message)
    messages.append(
        {"role": "tool", "content": f"{resultat_outil}"}
    )

    return messages



messages = [
    {"role": "user", "content": f"Quelle météo il fera demain à Laval (Québec) ?"}
    # {"role": "user", "content": f"Quelle météo il fera demain à Kuujjuaq (Québec) ?"}
]

response = chat(model="MODEL_COURANT", messages=messages, tools=outils, think=False, keep_alive=3000)


if response.message.tool_calls:
    messages = utiliser_outils(response, messages)
    response = chat(model="MODEL_COURANT", messages=messages, tools=outils, think=False, keep_alive=3000)


pprint(response.model_dump())

### Exercice 8.4 - Utilisation de plusieurs outils 

Qu'arrive-t-il si plusieurs outils doivent être utilisés pour le même prompt ?

Le comportement de Gemma n'est pas déterministe dans ce cas. Parfois, il renvoie les deux outils à utiliser dans la première réponse, mais parfois il s'attend à plusieurs itérations. Par exemple, il demande la date en premier, puis dans la deuxième réponse, il demande la météo, puis, dans une troisième, il donne sa réponse.

Exécuter le code suivant plusieurs fois, pour voir les différents comportements.

In [ ]:
messages = [
        {"role": "user", "content": f"Quelle météo il fera demain à Laval (Québec) ? Et indique moi la date aussi"}
    ]

response = chat(model="MODEL_COURANT", messages=messages, tools=outils, think=False, keep_alive=3000)


if response.message.tool_calls:
    messages = utiliser_outils(response, messages)
    response = chat(model="MODEL_COURANT", messages=messages, tools=outils, think=False, keep_alive=3000)


pprint(response.model_dump())

#### Exercice 8.4.1 - Boucle while

Ici, on change seulement le `if` par un `while`. Donc, tant que le modèle nécessite des outils, on les utilisent.

Donc, dans notre exemple, l'appel final à Gemma comprendra plusieurs messages:

1. Question initiale de l'utilisateur
2. Demande d'utilisation de l'outil Date par Gemma
3. Réponse de l'outil Date
4. Demande d'utilisation de l'outil Meteo par Gemma
3. Réponse de l'outil Météo
5. Réponse finale de Gemma à la question initiale

In [ ]:
messages = [
        {"role": "user", "content": f"Quelle météo il fera demain à Laval (Québec) ? Et indique moi la date aussi"}
    ]

response = chat(model="MODEL_COURANT", messages=messages, tools=outils, think=False, keep_alive=3000)


while response.message.tool_calls:
    messages = utiliser_outils(response, messages)
    response = chat(model="MODEL_COURANT", messages=messages, tools=outils, think=False, keep_alive=3000)

pprint(response.model_dump())

### Exercice 8.5 - Augmentation de la complexité

Si on demande la date actuelle à Gemma sans outil, il va nous répondre quelque chose de non cohérent. Il "pense" qu'il connait cette date.

Donc, si la demande est plus subtile, il est possible qu'il décide de ne pas utiliser tous les outils qu'il devrait avoir besoin pour répondre.

In [ ]:
messages = [
        {"role": "user", "content": f"Quelle météo il fera demain à Montréal (Québec) ? \
            Dis-moi si c'est dans les normales de saison"}
    ]

response = chat(model="MODEL_COURANT", messages=messages, tools=outils, think=False, keep_alive=3000)


while response.message.tool_calls:
    messages = utiliser_outils(response, messages)
    response = chat(model="MODEL_COURANT", messages=messages, tools=outils, think=False, keep_alive=3000)

pprint(response.model_dump())

#### Exercice 8.5.1 - Un peu d'aide

En utilisant le role `system`, on rappelle à Gemma d'utiliser les outils si c'est nécessaire.

Ce qui n'est pas toujours parfait...

In [ ]:
messages = [
        {"role": "system", "content": f"Si il te manque des informations pour répondre, n'hésite pas utiliser les outils pertinents."},
        {"role": "user", "content": f"Quelle météo il fera demain à Montréal (Québec) ? \
            Dis-moi si c'est dans les normales de saison pour la date actuelle"}
    ]

response = chat(model="MODEL_COURANT", messages=messages, tools=outils, think=False, keep_alive=3000)


while response.message.tool_calls:
    messages = utiliser_outils(response, messages)
    response = chat(model="MODEL_COURANT", messages=messages, tools=outils, think=False, keep_alive=3000)

pprint(response.model_dump())

## Exercice 9 - La suite

Le modèle n'est que le "moteur". On doit utiliser du code et des outils pour épauler ce modèle.

Par exemple, il serait pertinent d'ajouter un moteur de recherche web comme outil. Ce qui permettrait au modèle d'avoir accès à l'actualité, ou aux connaissances qui ont été réalisées depuis la création du modèle. 

Le modèle ne peut pas savoir qui a gagné la partie de hockey d'hier... mais avec une recherche, il pourrait s'informer.

Un autre exemple serait de garder en mémoire les paramètres d'utilisateur. Plus haut, nous avons vu comment garder une mémoire de conversation. Mais il serait utilise de gader une mémoire des préférences de l'utilisateur.

Donc, un outil qui sauvegarderait les informations pertinentes dans un fichier texte, permettrait d'assurer une persistance entre les séances d'utilisation. 

Qu'est-ce qu'il faudrait mettre dans ce fichier texte... C'est Gemma qui décidera !